In [ ]:
import pandas as pd
import numpy as np
import re
import ftfy
from pathlib import Path

In [ ]:
# Load file. Comment the line that doesn't work for you. csv or xlsx
df = pd.read_csv('data.csv', dtype=str) # for CSV
#df = pd.read_excel('data.xlsx', dtype=str) # for excel

df.head()

In [ ]:
# Specify which columns contain names, emails, phone numbers
NAME_COLUMN = "First Name"
EMAIL_COLUMN = "Email"
PHONE_COLUMN = "Phone"

In [ ]:
# Cleaning functions
def clean_text(value):
    if pd.isna(value):
        return value
    
    value = str(value)
    
    # Fix encoding issues
    value = ftfy.fix_text(value)
    
    # Remove leading/trailing whitespace
    value = re.sub(r'\s+', ' ', value)
    
    return value.strip()

def clean_email(email):
    
    if pd.isna(email):
        return email
    
    return str(email).strip().lower()

def clean_phone(phone):
    
    if pd.isna(phone):
        return ""
    
    phone = str(phone).strip()
    
    # Remove .0 from Excel numbers
    if phone.endswith(".0"):
        phone = phone[:-2]
        
    # Scientific notation
    try:
        if "e+" in phone.lower():
            phone = format(float(phone), '0f')
    except:
        pass
    
    # Keep digits only
    phone = re.sub(r"[^0-9]", "", phone)
    
    return phone

def split_name(name):
    
    if pd.isna(name):
        return pd.Series([None, None, None])
    
    name = clean_text(name)
    
    # Last, First change order
    if ',' in name:
        
        last, first = name.split(',', 1)
        
        first = first.strip().title()
        last = last.strip().title()
        
        return pd.Series([
            first,
            last,
            f"{first} {last}"
        ])
        
    # First Last
    parts = name.split()
    
    if len(parts) == 1:
        
        return pd.Series([
            parts[0].title(),
            '',
            parts[0].title()
        ])
        
    first_name = parts[0].title()
    last_name = ' '.join(parts[1:]).title()
    
    return pd.Series([
        first_name,
        last_name,
        f"{first_name} {last_name}"
    ])

In [ ]:
# Cleaning all text columns
for col in df.columns:
    
    if df[col].dtype == "object":
        
        df[col] = df[col].apply(clean_text)
        
# Emails
if EMAIL_COLUMN in df.columns:
    
    df[EMAIL_COLUMN] = df[EMAIL_COLUMN].apply(clean_email)
    
# Phone numbers
if PHONE_COLUMN in df.columns:
    
    df[PHONE_COLUMN] = df[PHONE_COLUMN].apply(clean_phone)
    
# Names
if NAME_COLUMN in df.columns:
    
    df[
        ["First Name",
         "Last Name",
         "Full Name Clean"]
    ] = df[NAME_COLUMN].apply(split_name)

In [ ]:
# Quality report
print("Rows:", len(df))

print("Missing Emails:",
      df[EMAIL_COLUMN].isna().sum())

print("Missing Phone Numbers:",
      df[PHONE_COLUMN].isna().sum())
print("Missing Names:", 
      df[NAME_COLUMN].isna().sum())

In [ ]:
# Save new file with cleaned data - Comment the line that doesn't work for you. csv or xlsx
df.to_csv(
    "new_cleaned_data.csv",
    index=False
)

#df.to_excel(
#    "new_cleaned_data.xlsx",
#    index=False
#)